# Lab

Build a service desk dispatcher with a binary heap for urgency and a chained map for owners. Lower urgency values come first; arrival numbers break ties. Each task extends the same components, and the last task coordinates them for registration, cancellation, and dispatch.

Each cell contains its own setup. Carry your completed methods forward explicitly. Use Edit to change a draft and Run to check it. Sign in before submitting; multiple submissions are allowed and the highest score is kept.

## 1. Insert Pending Tickets

```{index} Insert Pending Tickets
```

Complete `Enqueue`: append the tuple, then bubble up using the supplied comparer. Compare the whole tuple so arrival numbers resolve equal urgency. The root should be ticket 105, Count three, and the heap valid. Explain why the raw array is not the dispatch order.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var pending = new BinaryMinHeap<(int Urgency, long Arrival, int Id)>();
long arrival = 0;
foreach (var (id, urgency) in new[] { (101, 2), (105, 0), (109, 2) })
    pending.Enqueue((urgency, arrival++, id));
Console.WriteLine($"next={pending.Peek().Id}, count={pending.Count}, valid={pending.IsValid()}");
Console.WriteLine("raw IDs=" + string.Join(", ", pending.Snapshot().Select(t => t.Id)));

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var pending = new BinaryMinHeap<(int Urgency, long Arrival, int Id)>();
long arrival = 0;
foreach (var (id, urgency) in new[] { (101, 2), (105, 0), (109, 2) })
    pending.Enqueue((urgency, arrival++, id));
Console.WriteLine($"next={pending.Peek().Id}, count={pending.Count}, valid={pending.IsValid()}");
Console.WriteLine("raw IDs=" + string.Join(", ", pending.Snapshot().Select(t => t.Id)));

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

next=105, count=3, valid=True
raw IDs=105, 101, 109


## 2. Remove the Next Ticket

```{index} Remove the Next Ticket
```

Complete `Dequeue`. Preserve shape by removing the last position, handle a singleton without indexing an empty list, and sink the replacement toward the smaller child. Reuse your enqueue method.

Expected dispatch IDs: 105, 101, 109. Empty `TryDequeue` is false; after reuse, ticket 113 is removed and Count returns to zero. Check why choosing only the left child can violate heap order.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var pending = new BinaryMinHeap<(int Urgency, long Arrival, int Id)>();
long arrival = 0;
foreach (var (id, urgency) in new[] { (101, 2), (105, 0), (109, 2) })
    pending.Enqueue((urgency, arrival++, id));
while (pending.TryDequeue(out var ticket)) Console.WriteLine($"serve ID={ticket.Id}");
Console.WriteLine($"empty success={pending.TryDequeue(out _)}");
pending.Enqueue((0, arrival++, 113));
Console.WriteLine($"reused ID={pending.Dequeue().Id}, count={pending.Count}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var pending = new BinaryMinHeap<(int Urgency, long Arrival, int Id)>();
long arrival = 0;
foreach (var (id, urgency) in new[] { (101, 2), (105, 0), (109, 2) })
    pending.Enqueue((urgency, arrival++, id));
while (pending.TryDequeue(out var ticket)) Console.WriteLine($"serve ID={ticket.Id}");
Console.WriteLine($"empty success={pending.TryDequeue(out _)}");
pending.Enqueue((0, arrival++, 113));
Console.WriteLine($"reused ID={pending.Dequeue().Id}, count={pending.Count}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

serve ID=105
serve ID=101
serve ID=109
empty success=False
reused ID=113, count=0


## 3. Store Owners Despite Collisions

```{index} Store Owners Despite Collisions
```

Complete `Put` and `TryGetValue`. IDs 101, 105, and 109 collide in bucket one of the initial four-bucket map. Replacement must change the owner without adding another key or changing Count.

Expected replacement result false, Count three, owner Lee for 105, and missing lookup false for 113. Compare IDs inside the selected chain; sharing a bucket does not make two IDs equal.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
owners.Put(101, "Amina"); owners.Put(105, "Diego"); owners.Put(109, "Mina");
Console.WriteLine($"replacement inserted={owners.Put(105, "Lee")}, count={owners.Count}");
Console.WriteLine($"105 found={owners.TryGetValue(105, out string owner)}, owner={owner}");
Console.WriteLine($"missing found={owners.TryGetValue(113, out _)}");
Console.WriteLine(owners.DescribeBuckets());

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool TryGetValue(int id, out string owner)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
owners.Put(101, "Amina"); owners.Put(105, "Diego"); owners.Put(109, "Mina");
Console.WriteLine($"replacement inserted={owners.Put(105, "Lee")}, count={owners.Count}");
Console.WriteLine($"105 found={owners.TryGetValue(105, out string owner)}, owner={owner}");
Console.WriteLine($"missing found={owners.TryGetValue(113, out _)}");
Console.WriteLine(owners.DescribeBuckets());

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

replacement inserted=False, count=3
105 found=True, owner=Lee
missing found=False
0:  | 1: 101, 105, 109 | 2:  | 3: 


## 4. Rehash and Cancel without Losing Owners

```{index} Rehash and Cancel without Losing Owners
```

Complete `Grow` and `Remove`, using your map methods from Task 3. Recompute each old key's bucket in the doubled array. Growth must preserve Count; removing one key must leave its collision partners reachable.

Expected capacity eight, Count four before removal, and Count three afterward. ID 109 still belongs to Mina; removing an absent ID returns false. Draw the old/new positions for 101, 105, 109, and 113.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
foreach (var (id, assignedOwner) in new[] { (101, "Amina"), (105, "Diego"), (109, "Mina"), (113, "Noor") })
    owners.Put(id, assignedOwner);
Console.WriteLine($"buckets={owners.BucketCount}, count={owners.Count}");
Console.WriteLine(owners.DescribeBuckets());
Console.WriteLine($"remove 101={owners.Remove(101)}, missing={owners.Remove(999)}");
Console.WriteLine($"109 found={owners.TryGetValue(109, out string owner)}, owner={owner}, count={owners.Count}");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    private void Grow()
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
foreach (var (id, assignedOwner) in new[] { (101, "Amina"), (105, "Diego"), (109, "Mina"), (113, "Noor") })
    owners.Put(id, assignedOwner);
Console.WriteLine($"buckets={owners.BucketCount}, count={owners.Count}");
Console.WriteLine(owners.DescribeBuckets());
Console.WriteLine($"remove 101={owners.Remove(101)}, missing={owners.Remove(999)}");
Console.WriteLine($"109 found={owners.TryGetValue(109, out string owner)}, owner={owner}, count={owners.Count}");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

buckets=8, count=4
0:  | 1: 105, 113 | 2:  | 3:  | 4:  | 5: 101, 109 | 6:  | 7: 
remove 101=True, missing=False
109 found=True, owner=Mina, count=3


## 5. Coordinate Registration and Dispatch

```{index} Coordinate Registration and Dispatch
```

Use your heap and map methods from Tasks 1–4. Complete registration and the dispatch loop. Reject every previously registered ID in this batch using `seen`; cancelled or served IDs cannot be reused. This keeps stale entries distinguishable by absence from the owner map.

Cancel urgent ticket 105 before dispatch. Expected output: duplicate accepted false; skip cancelled 105; serve 101/Amina, then 109/Mina; both active and pending Counts zero. Explain why active and pending Counts differ immediately after cancellation. Include an empty-queue check and confirm that equal urgency preserves arrival order.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
var seen = new HashSet<int>();
var pending = new BinaryMinHeap<(int Urgency, long Arrival, int Id)>();
long arrival = 0;
bool Register(int id, string owner, int urgency)
{
    // Your code starts here.
    /* TODO: reject every previously registered ID, then add both live indexes. */
    // Your code ends here.
    }
Register(101, "Amina", 2); Register(105, "Diego", 0); Register(109, "Mina", 2);
Console.WriteLine($"duplicate accepted={Register(101, "Other", 0)}");
owners.Remove(105);
// Your code starts here.
/* TODO: dispatch while skipping cancelled IDs; remove every served owner record. */
// Your code ends here.
Console.WriteLine($"active={owners.Count}, pending={pending.Count}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
var seen = new HashSet<int>();
var pending = new BinaryMinHeap<(int Urgency, long Arrival, int Id)>();
long arrival = 0;
bool Register(int id, string owner, int urgency)
{
    if (!seen.Add(id)) return false;
    owners.Put(id, owner);
    pending.Enqueue((urgency, arrival++, id));
    return true;
}
Register(101, "Amina", 2); Register(105, "Diego", 0); Register(109, "Mina", 2);
Console.WriteLine($"duplicate accepted={Register(101, "Other", 0)}");
owners.Remove(105);
while (pending.TryDequeue(out var ticket))
{
    if (!owners.TryGetValue(ticket.Id, out string owner))
    {
        Console.WriteLine($"skip cancelled {ticket.Id}");
        continue;
    }
    owners.Remove(ticket.Id);
    Console.WriteLine($"serve {ticket.Id}: {owner}");
}
Console.WriteLine($"active={owners.Count}, pending={pending.Count}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

duplicate accepted=False
skip cancelled 105
serve 101: Amina
serve 109: Mina
active=0, pending=0


## Submit

```{index} lab submission
```

Use the assignment submit panel when it is available in the published book.